# INITIAL FILTERING

## Imports

In [1]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import requests
import rioxarray
import earthaccess

RAW = Path("../data/raw")
OUT = Path("../data/processed")
FRONT = Path("../public/data")
for d in (RAW / "worldclim", RAW / "modis_lc", OUT, FRONT):
    d.mkdir(parents=True, exist_ok=True)

RES = 0.5
NCOLS = int(360 / RES)            # 720

/home/yoga/miniforge3/envs/spaceapps/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data download (skips files already present)

In [2]:
def download(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f"✓ already have {dest.name}")
        return dest
    print(f"↓ downloading {dest.name} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
    return dest

# WorldClim 2.1 bioclim, 10 arc-minutes
WC_URL = "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_bio.zip"
wc_zip = download(WC_URL, RAW / "worldclim" / "wc2.1_10m_bio.zip")
wc_bio1 = RAW / "worldclim" / "wc2.1_10m_bio_1.tif"
if not wc_bio1.exists():
    with zipfile.ZipFile(wc_zip) as z:
        z.extractall(RAW / "worldclim")
    print("✓ unzipped WorldClim")

# MODIS MCD12C1 land cover (NASA Earthdata login needed)
existing_lc = sorted((RAW / "modis_lc").glob("MCD12C1*.hdf"))
if existing_lc:
    lc_path = existing_lc[-1]
    print(f"✓ already have {lc_path.name}")
else:
    earthaccess.login(persist=True)
    res = earthaccess.search_data(short_name="MCD12C1", version="061",
                                  temporal=("2022-01-01", "2022-12-31"))
    lc_path = Path(earthaccess.download(res[:1], str(RAW / "modis_lc"))[0])
    print(f"✓ downloaded {lc_path.name}")

✓ already have wc2.1_10m_bio.zip
✓ already have MCD12C1.A2022001.061.2023244164746.hdf


## Stable cell ID

In [3]:
def cell_id(lat, lon):
    lat = np.asarray(lat); lon = np.asarray(lon)
    row = ((90 - lat) // RES).astype(int)       # 0 = northernmost row
    col = ((lon + 180) // RES).astype(int)      # 0 = lon -180
    return row * NCOLS + col

## Base grid: WorldClim temperature (defines land cells)

In [4]:
t = rioxarray.open_rasterio(f"{RAW}/worldclim/wc2.1_10m_bio_1.tif", masked=True).squeeze()
t05 = t.coarsen(x=3, y=3, boundary="trim").mean().rio.write_crs(4326)

## MODIS land cover (% per IGBP class)

In [5]:
with rasterio.open(lc_path) as src:
    sds = next(s for s in src.subdatasets if s.endswith("Land_Cover_Type_1_Percent"))

pct = rioxarray.open_rasterio(sds, masked=True).squeeze()
print("Land cover bands:", dict(pct.sizes))     # expect band: 17 → band = class + 1

def igbp(c):
    """Percent of IGBP class c per 0.5° cell, aligned to t05."""
    return (pct.sel(band=c + 1)
               .coarsen(x=10, y=10, boundary="trim").mean()
               .rio.write_crs(4326)
               .rio.reproject_match(t05))

layers = {
    "water_pct":  igbp(0),
    "forest_pct": sum(igbp(c) for c in range(1, 6)),
    "urban_pct":  igbp(13),
    "snow_pct":   igbp(15),     # info only
    "barren_pct": igbp(16),     # info only
}

/home/yoga/miniforge3/envs/spaceapps/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Land cover bands: {'band': 17, 'y': 3600, 'x': 7200}


## Master table: one row per land cell

In [6]:
lon2d, lat2d = np.meshgrid(t05.x.values, t05.y.values)
cells = pd.DataFrame({"lat": lat2d.ravel(), "lon": lon2d.ravel(),
                      "temp": t05.values.ravel()})
for name, da in layers.items():
    cells[name] = da.values.ravel()

cells = cells.dropna(subset=["temp"]).reset_index(drop=True)   # land only
cells.insert(0, "cell_id", cell_id(cells.lat, cells.lon))
cells["temp_p"] = cells["temp"].rank(pct=True)
print(f"Land cells: {len(cells):,}")

Land cells: 95,447


## Filter registry

In [7]:
FILTERS = {
    "forest": lambda df: df.forest_pct.fillna(0) <= 10,
    "urban":  lambda df: df.urban_pct.fillna(0)  <= 2,
    "water":  lambda df: df.water_pct.fillna(0)  <= 25,
    # add later, e.g.: "protected": lambda df: df.protected_share.fillna(0) < 0.5,
}

def apply_filters(df):
    pass_cols = []
    for name, rule in FILTERS.items():
        col = f"pass_{name}"
        df[col] = rule(df)
        pass_cols.append(col)
    df["passes_all"] = df[pass_cols].all(axis=1)
    df["excluded_by"] = df[pass_cols].apply(
        lambda r: ", ".join(c[5:] for c in pass_cols if not r[c]), axis=1)
    return df

cells = apply_filters(cells)

## Funnel (for the pitch)

In [8]:
print("\nFunnel:")
remaining = np.ones(len(cells), dtype=bool)
print(f"  start     : {remaining.sum():,} cells")
for name in FILTERS:
    remaining &= cells[f"pass_{name}"].values
    print(f"  after {name:<6}: {remaining.sum():,} cells")


Funnel:
  start     : 95,447 cells
  after forest: 77,604 cells
  after urban : 75,129 cells
  after water : 62,626 cells


## Validation: do known analogs survive?

In [9]:
analogs = pd.DataFrame([
    ("Atacama Desert",        -24.50,  -69.25),
    ("Antarctic Dry Valleys", -77.50,  162.00),
    ("Haughton Crater",        75.38,  -89.67),
    ("Mauna Kea",              19.82, -155.47),
    ("Black Point Lava Flow",  35.70, -111.40),
    ("Rio Tinto",              37.70,   -6.60),
    ("Teide",                  28.27,  -16.64),
], columns=["name", "lat", "lon"])
analogs["cell_id"] = cell_id(analogs.lat, analogs.lon)

check = analogs.merge(
    cells[["cell_id", "forest_pct", "urban_pct", "water_pct", "passes_all", "excluded_by"]],
    on="cell_id", how="left")
check["passes_all"] = check["passes_all"].fillna(False)
check["excluded_by"] = check["excluded_by"].fillna("not a land cell")
print("\nKnown analogs:")
display(check[["name", "forest_pct", "urban_pct", "water_pct", "passes_all", "excluded_by"]].round(1))



Known analogs:


,name,forest_pct,urban_pct,water_pct,passes_all,excluded_by
0,Atacama Desert,0.0,0.0,0.0,True,
1,Antarctic Dry Valleys,0.0,0.0,0.0,True,
2,Haughton Crater,0.0,0.0,0.0,True,
3,Mauna Kea,35.0,0.2,13.5,False,forest
4,Black Point Lava Flow,0.0,0.0,0.0,True,
5,Rio Tinto,5.0,0.7,0.0,True,
6,Teide,2.8,1.6,53.0,False,water


## Save master table

In [10]:
cells.to_parquet(OUT / "cells.parquet", index=False)
print(f"\nSaved {OUT / 'cells.parquet'}")


Saved ../data/processed/cells.parquet


## Frontend export

In [11]:
FEATURES = ["temp_p"]          # add ndvi_p, aridity_p, rough_p, ... as they come

ok = cells[cells.passes_all]
front = {"cell_id": ok.cell_id.tolist(),
         "lat": ok.lat.round(2).tolist(),
         "lon": ok.lon.round(2).tolist()}
for f in FEATURES:
    front[f] = ok[f].round(3).tolist()

with open(FRONT / "cells.json", "w") as fh:
    json.dump(front, fh, separators=(",", ":"))

bad = cells[~cells.passes_all]
with open(FRONT / "excluded.json", "w") as fh:
    json.dump({"cell_id": bad.cell_id.tolist(),
               "excluded_by": bad.excluded_by.tolist()}, fh, separators=(",", ":"))

print(f"Exported cells.json ({len(ok):,} cells) and excluded.json ({len(bad):,} cells)")

Exported cells.json (62,626 cells) and excluded.json (32,821 cells)
